In [18]:
import pymongo
from pymongo import MongoClient
import pandas as pd

try: 
    client = MongoClient("localhost", 5001)
    print("Connected successfully!!!") 
except:
    print("Could not connect to MongoDB")

data = client["dataset_db"]["fine_tuning"]

Connected successfully!!!


In [19]:
query = {}

cursor = data.find(query)

df = pd.DataFrame(list(cursor))

In [20]:
df

,_id,instruction,writing_intention,before_text,after_text,diff_array,start_idx,end_idx
0,6625f76aa065f639e31bd6d8,INSTRUCTION FOR WRITING INTENTION,Discourse Planning,\section{Constructing ``TenPageStories'' Datas...,\section{Constructing ``TenPageStories'' Datas...,"[[0, \section{Constructing ``TenPageStories'' ...",0,2
1,6625f76aa065f639e31bd6d9,INSTRUCTION FOR WRITING INTENTION,Discourse Planning,\section{Conclusion}\label{sec:conclusion}\n,\section{Conclusion}\label{sec:conclusion}\n,"[[0, ], [4, \section{Conclusion}\label{sec:con...",3,3
2,6625f76aa065f639e31bd6da,INSTRUCTION FOR WRITING INTENTION,Discourse Planning,\section{Conclusion}\label{sec:conclusion}\n,\section{Limitation}\label{sec:limitation}\n\n,"[[0, \section{Limitation}\label{sec:limitation...",4,7
3,6625f76aa065f639e31bd6db,INSTRUCTION FOR WRITING INTENTION,Discourse Planning,% This must be in the first 5 lines to tell ar...,% This must be in the first 5 lines to tell ar...,"[[0, % This must be in the first 5 lines to te...",22,25
4,6625f76aa065f639e31bd6dc,INSTRUCTION FOR WRITING INTENTION,Discourse Planning,\State \texttt{prompt} $\gets$ \texttt...,\State \texttt{prompt} $\gets$ \texttt...,"[[0, \State \texttt{prompt} $\gets$ \t...",265,266
...,...,...,...,...,...,...,...,...
275,6625f76aa065f639e31bd7eb,INSTRUCTION FOR WRITING INTENTION,Lexical,\begin{figure}[!ht]\n\centering\n% <left> <low...,\begin{figure}[!ht]\n\centering\n% <left> <low...,"[[0, \begin{figure}[!ht]\n\centering\n% <left>...",10279,10286
276,6625f76aa065f639e31bd7ec,INSTRUCTION FOR WRITING INTENTION,Lexical,\begin{figure}[!ht]\n\centering\n% <left> <low...,\begin{figure}[!ht]\n\centering\n% <left> <low...,"[[0, \begin{figure}[!ht]\n\centering\n% <left>...",10289,10299
277,6625f76aa065f639e31bd7ed,INSTRUCTION FOR WRITING INTENTION,Visual,% $\text{GAT}_{\text{Motif}}$ & 0.00 & 0.00 ...,% $\text{GAT}_{\text{Motif}}$ & 0.00 & 0.00 ...,"[[0, % $\text{GAT}_{\text{Motif}}$ & 0.00 & ...",10317,10317
278,6625f76aa065f639e31bd7ee,INSTRUCTION FOR WRITING INTENTION,Visual,"In this section, we report F1 scores for the b...","In this section, we report F1 scores for the b...","[[0, In this section, we report F1 scores for ...",10323,10324


In [21]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "mistralai/Mixtral-8x7B-Instruct-v0.1"
tokenizer = AutoTokenizer.from_pretrained(model_id)

#model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto")

def tokenize(data):
  msg = [
      {"role": "user", "content": data}
  ]

  tokenized = tokenizer.apply_chat_template(msg, return_tensors="pt")
  #print("before size", before_tokenized.shape, "\nafter_size:", after_tokenized.shape)

  return tokenized.shape[1]

In [22]:
df["before_token_count"] = df["before_text"].map(tokenize)
df["after_token_count"] = df["after_text"].map(tokenize)

In [23]:
df

,_id,instruction,writing_intention,before_text,after_text,diff_array,start_idx,end_idx,before_token_count,after_token_count
0,6625f76aa065f639e31bd6d8,INSTRUCTION FOR WRITING INTENTION,Discourse Planning,\section{Constructing ``TenPageStories'' Datas...,\section{Constructing ``TenPageStories'' Datas...,"[[0, \section{Constructing ``TenPageStories'' ...",0,2,63,91
1,6625f76aa065f639e31bd6d9,INSTRUCTION FOR WRITING INTENTION,Discourse Planning,\section{Conclusion}\label{sec:conclusion}\n,\section{Conclusion}\label{sec:conclusion}\n,"[[0, ], [4, \section{Conclusion}\label{sec:con...",3,3,22,22
2,6625f76aa065f639e31bd6da,INSTRUCTION FOR WRITING INTENTION,Discourse Planning,\section{Conclusion}\label{sec:conclusion}\n,\section{Limitation}\label{sec:limitation}\n\n,"[[0, \section{Limitation}\label{sec:limitation...",4,7,22,23
3,6625f76aa065f639e31bd6db,INSTRUCTION FOR WRITING INTENTION,Discourse Planning,% This must be in the first 5 lines to tell ar...,% This must be in the first 5 lines to tell ar...,"[[0, % This must be in the first 5 lines to te...",22,25,1044,1053
4,6625f76aa065f639e31bd6dc,INSTRUCTION FOR WRITING INTENTION,Discourse Planning,\State \texttt{prompt} $\gets$ \texttt...,\State \texttt{prompt} $\gets$ \texttt...,"[[0, \State \texttt{prompt} $\gets$ \t...",265,266,1163,1164
...,...,...,...,...,...,...,...,...,...,...
275,6625f76aa065f639e31bd7eb,INSTRUCTION FOR WRITING INTENTION,Lexical,\begin{figure}[!ht]\n\centering\n% <left> <low...,\begin{figure}[!ht]\n\centering\n% <left> <low...,"[[0, \begin{figure}[!ht]\n\centering\n% <left>...",10279,10286,2175,2178
276,6625f76aa065f639e31bd7ec,INSTRUCTION FOR WRITING INTENTION,Lexical,\begin{figure}[!ht]\n\centering\n% <left> <low...,\begin{figure}[!ht]\n\centering\n% <left> <low...,"[[0, \begin{figure}[!ht]\n\centering\n% <left>...",10289,10299,2245,2158
277,6625f76aa065f639e31bd7ed,INSTRUCTION FOR WRITING INTENTION,Visual,% $\text{GAT}_{\text{Motif}}$ & 0.00 & 0.00 ...,% $\text{GAT}_{\text{Motif}}$ & 0.00 & 0.00 ...,"[[0, % $\text{GAT}_{\text{Motif}}$ & 0.00 & ...",10317,10317,2099,2099
278,6625f76aa065f639e31bd7ee,INSTRUCTION FOR WRITING INTENTION,Visual,"In this section, we report F1 scores for the b...","In this section, we report F1 scores for the b...","[[0, In this section, we report F1 scores for ...",10323,10324,2148,2152


In [24]:
df["before_token_count"].describe()

count     280.000000
mean     1347.078571
std       557.817220
min        22.000000
25%      1009.000000
50%      1438.000000
75%      1740.500000
max      2452.000000
Name: before_token_count, dtype: float64

In [25]:

df["after_token_count"].describe()

count     280.000000
mean     1352.539286
std       554.693953
min        22.000000
25%      1019.250000
50%      1440.000000
75%      1741.500000
max      2522.000000
Name: after_token_count, dtype: float64

In [26]:
from nltk.metrics.distance import *

def get_diff(data):
  return edit_distance(data["before_text"], data["after_text"])